# Development data exploration
Inspect the ordinary clinical-photo manifest, missingness, class balance, and group leakage before training. This notebook never accesses DDI.

## Load and validate the development manifest
Run from the repository root. Create the manifest with the shared schema and group-safe split utilities before continuing.

In [ ]:
import pandas as pd
from src.data.datasets import validate_manifest
from src.data.splits import class_coverage_report, leakage_report
manifest = validate_manifest(pd.read_csv('data/processed/development_manifest.csv'))
assert not manifest.dataset.fillna('').str.upper().eq('DDI').any()
assert leakage_report(manifest).empty
display(manifest.groupby(['dataset', 'split']).size().unstack(fill_value=0))
display(manifest.isna().mean().sort_values(ascending=False).rename('missing_fraction'))
display(manifest[['harmonized_diagnosis', 'binary_target', 'lesion_present', 'normal_skin']].value_counts(dropna=False))
display(class_coverage_report(manifest, 'binary_target', ('train', 'validation', 'test')))

## Inspect deterministic and augmented views
Use the shared transforms; training augmentation is stochastic, while validation/test preprocessing is deterministic.

In [ ]:
from src.data.transforms import build_transforms
train_transform = build_transforms(224, training=True)
evaluation_transform = build_transforms(224, training=False)
# Display representative source and transformed images before launching experiments.